<a href="https://colab.research.google.com/github/Cousar/Colab-codes-for-TE-project/blob/main/1D_Xshift_HalfBlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from google.colab import drive
from scipy.special import erf
from openpyxl.chart import ScatterChart, Reference, Series

drive.mount("/content/drive")

file_path = "/content/drive/Othercomputers/My Laptop/Germany/Germany main files/Essen/Ph.D/1.13-FA Cup Project/1.12-Analytical/RayOptic.xlsx"

RayOptic = pd.read_excel(file_path, sheet_name="FixedBeam")

def erf_Pnumbers_Xshift(mu, sigma):
    return 0.5 * (1 + erf(mu / sigma / np.sqrt(2)))

mu = np.arange(-20, 21, 1)

with pd.ExcelWriter(file_path, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:

    for index, row in RayOptic.iterrows():

        energy = row["Energy_MeV"]
        sigma_value = row["XspotSize_mm"]

        share_values = erf_Pnumbers_Xshift(mu, sigma_value)

        df = pd.DataFrame({
            "Energy_MeV": energy,
            "sigma": sigma_value,
            "mu": mu,
            "share": share_values
        })

        sheet_name = f"sigma_{sigma_value:.2f}"

        df.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.sheets[sheet_name]

        chart = ScatterChart()
        chart.title = f"mu vs share, sigma={sigma_value:.2f}"
        chart.x_axis.title = "mu"
        chart.y_axis.title = "share"

        # Chart size
        chart.width = 22
        chart.height = 11

        # X-axis range
        chart.x_axis.scaling.min = -20
        chart.x_axis.scaling.max = 20
        chart.x_axis.majorUnit = 5

        # Y-axis range
        chart.y_axis.scaling.min = 0
        chart.y_axis.scaling.max = 1
        chart.y_axis.majorUnit = 0.1

        # Force axes to be visible
        chart.x_axis.delete = False
        chart.y_axis.delete = False

        # Move axes to borders
        chart.y_axis.crosses = "min"
        chart.x_axis.crosses = "min"

        # Show axis numbers
        chart.x_axis.tickLblPos = "nextTo"
        chart.y_axis.tickLblPos = "nextTo"

        chart.x_axis.numFmt = "0"
        chart.y_axis.numFmt = "0.0"

        chart.x_axis.majorTickMark = "out"
        chart.y_axis.majorTickMark = "out"

        # Data range
        xvalues = Reference(ws, min_col=3, min_row=2, max_row=len(df) + 1)  # mu
        yvalues = Reference(ws, min_col=4, min_row=2, max_row=len(df) + 1)  # share

        series = Series(yvalues, xvalues, title="share")

        # Discrete black points only
        series.marker.symbol = "circle"
        series.marker.size = 2

        # Remove connecting line
        series.graphicalProperties.line.noFill = True

        # Black marker color
        series.marker.graphicalProperties.solidFill = "000000"
        series.marker.graphicalProperties.line.solidFill = "000000"

        chart.series.append(series)

        # Remove legend
        chart.legend = None

        # Add chart to sheet
        ws.add_chart(chart, "F2")

print("DONE")

MessageError: Error: credential propagation was unsuccessful